# ERA5 geospatial point extraction (archived research notebook)

This notebook documents previously explored Earth Engine steps. Cloud resources have been replaced by `EE_PROJECT_ID` and `EE_EXPORT_BUCKET` environment variables. Large input data and credentials are not included; this public copy has not been rerun end-to-end. Run from the `scripts` directory.


In [ ]:
import ee
import pandas as pd

import os
ee.Initialize(project=os.environ["EE_PROJECT_ID"])

print(
    "EE OK. ERA5 size =",
    ee.ImageCollection("ECMWF/ERA5/HOURLY").size().getInfo()
)

In [ ]:
import pandas as pd
import ee
from pyprojroot import here

# 'here' ä¼èªå¨æ¾å°é¡¹ç®æ ¹ç®å½ (codes/)
# ç¶åä½ å°±å¯ä»¥ç´æ¥åç¸å¯¹äºæ ¹ç®å½çè·¯å¾ï¼ä¸ç¨ç®¡ ../../ è¿ç§å±çº§
from pathlib import Path
STATION_CSV = Path("../dataset/outcomes/from_s2/nmdb_selected_stations_latlon.csv")

# è¯»å csv
stations_df = pd.read_csv(STATION_CSV)

stations_ee = ee.FeatureCollection([
    ee.Feature(
        ee.Geometry.Point(row["lon"], row["lat"]),
        {"station": row["station"]}
    )
    for _, row in stations_df.iterrows()
])

print("Number of stations:", stations_ee.size().getInfo())

In [ ]:
ERA5_VARS = [
    "u_component_of_wind_10m",
    "v_component_of_wind_10m",
    "temperature_2m",
    "dewpoint_temperature_2m",
    "boundary_layer_height",
    "mean_boundary_layer_dissipation",
    "mean_total_precipitation_rate"
]

In [ ]:
def get_era5_year(year):
    start = f"{year}-01-01"
    end   = f"{year + 1}-01-01"

    return (
        ee.ImageCollection("ECMWF/ERA5/HOURLY")
        .filterDate(start, end)
        .select(ERA5_VARS)
    )


In [ ]:
def extract_points(ic, points):
    def extract_one(img):
        date = ee.Date(img.get("system:time_start")).format("YYYY-MM-dd HH:mm")

        fc = img.reduceRegions(
            collection=points,
            reducer=ee.Reducer.first(),
            scale=10000
        )

        return fc.map(lambda f: f.set("datetime", date))

    return ic.map(extract_one).flatten()


In [ ]:
# Monitoring example; only run after creating an Earth Engine export task.
# import time
# 
# while task.active():
#     print("Task running...")
#     time.sleep(30)
# 
# print("Task finished with state:", task.status())
# 

In [ ]:
for YEAR in range(2000, 2027): 
    era5_ic = get_era5_year(YEAR)
    table = extract_points(era5_ic, stations_ee)

    task = ee.batch.Export.table.toCloudStorage(
        collection=table,
        description=f"ERA5_points_{YEAR}",
        bucket=os.environ["EE_EXPORT_BUCKET"],
        fileNamePrefix=f"era5_points_{YEAR}",
        fileFormat="CSV"
    )

    task.start()
    print(f"Started export for {YEAR}")

In [ ]:
tasks = ee.batch.Task.list()

for t in tasks:
    s = t.status()
    desc = s.get("description", "")
    if desc.startswith("ERA5_points_"):
        print(desc, "->", s["state"])


In [ ]:
tasks = ee.batch.Task.list()

summary = {"READY": [], "RUNNING": [], "COMPLETED": [], "FAILED": []}

for t in tasks:
    s = t.status()
    name = s.get("description", "")
    if name.startswith("ERA5_points_"):
        summary[s["state"]].append(name)

for k, v in summary.items():
    print(f"{k}: {len(v)}")


In [ ]:
# Install in your terminal: pip install google-cloud-storage


## Downloading csv from GCS

In [ ]:
from google.cloud import storage
from pathlib import Path

# ====== éç½® ======
PROJECT_ID = os.environ["EE_PROJECT_ID"]
BUCKET_NAME = os.environ["EE_EXPORT_BUCKET"]
LOCAL_DIR = Path("../dataset/outcomes/from_s3")   # ç¸å¯¹è·¯å¾ï¼ç±»ä¼¼ here::hereï¼

LOCAL_DIR.mkdir(parents=True, exist_ok=True)

# ====== åå§å client ======
client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

# ====== ååºå¹¶ä¸è½½ ======
blobs = bucket.list_blobs()

csv_blobs = [b for b in blobs if b.name.endswith(".csv")]

print(f"Found {len(csv_blobs)} CSV files in bucket.")

for blob in csv_blobs:
    local_path = LOCAL_DIR / Path(blob.name).name
    blob.download_to_filename(local_path)
    print(f"Downloaded: {local_path}")

print("â All ERA5 CSV files downloaded.")


In [ ]:
from pathlib import Path

p = Path("../dataset/outcomes/from_s3")
print(p.exists())
print(list(p.glob("*.csv"))[:5])
